# HB model: bounded multi-start fit, all 12 subjects

Picks up from `data_preparation_and_hb_model.ipynb`'s Section 9 (see `HB_model_handoff.md`).
That notebook found the `k_prior` upper bound of 150 was too tight for subject 4
(pinned in 9/10 runs); a follow-up run with the bound raised to 1000 found the true
value settles around 190-198, with the fit essentially flat above ~150. This notebook
scales the same bounded multi-start procedure to all 12 subjects.

**Update after the first full attempt:** more than half of the first 9 subjects
fitted (1, 5, 6, 7, 9) hit the `k_prior` ceiling, meaning `K_PRIOR_UPPER=500` was
too tight generally, not just an edge case. Raised to 1000 for all subjects, and
`N_STARTS` dropped from 10 to 8 (still well-validated -- results have consistently
clustered on 8-9 of 10 starts landing on nearly the same solution) to offset some
of the added cost of a full re-run.

**Runtime:** each subject's 10-start fit took ~15-25 min for subject 4 (varies with
trial count per subject). All 12 subjects is on the order of 3-5 hours total.
This notebook checkpoints after every subject (saved to `/kaggle/working/hb_fits/`),
so if the session gets interrupted, re-running this notebook skips subjects already
done and picks up where it left off.

**Before running:** in Kaggle's notebook settings, turn Internet ON (Settings > Internet),
since Cell 2 clones the public data repo.

## Two ways to run this

**Option A - one big commit.** Save Version -> Save & Run All (Commit) once,
for all 12 subjects. Simpler, and Kaggle's ~9-12hr batch limit comfortably
covers the ~5hr estimate. This is a real background job on Kaggle's servers,
not an interactive session, so it survives you closing the tab.

**Option B - chunked commits (safer, a bit more manual work).** Split the 12
subjects into a few smaller commits (Section 4a/4b below), so a problem in one
chunk doesn't cost you the whole run. Recommended if you'd rather not trust
one long uninterrupted job, or if you're not confident your Kaggle plan
reliably gives you a full uninterrupted 9-12hr batch window.

Either way: **use Save & Run All (Commit), not the interactive "Run" button.**
Interactive editing sessions are ephemeral (they time out after ~1hr idle and
don't reliably survive refreshes), which is what caused lost progress before.


In [ ]:
import os
import time
import pickle
from pathlib import Path
from collections import deque

import numpy as np
import pandas as pd
from scipy.special import i0e
from scipy.optimize import minimize

OUT_DIR = Path("/kaggle/working/hb_fits")
if not OUT_DIR.parent.exists():
    OUT_DIR = Path("./hb_fits")   # fallback if not running on Kaggle
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Checkpoints will be saved to: {OUT_DIR.resolve()}")


## 1. Data loading and trial-level dataset

Looks for `data01_direction4priors.csv` among your uploaded Kaggle inputs first
(add it via "Add Input" > "Upload" in the right panel -- no internet needed).
Falls back to cloning the public repo only if the file isn't found locally
and internet is enabled for this session (Settings > Internet > On; Kaggle
also requires a verified phone number on your account for this to work).


In [ ]:
import glob

candidates = glob.glob("/kaggle/input/**/data01_direction4priors.csv", recursive=True)
candidates += glob.glob("./**/data01_direction4priors.csv", recursive=True)

if candidates:
    CSV_PATH = Path(candidates[0])
    print(f"Found uploaded file at: {CSV_PATH}")
else:
    print("No uploaded copy found, trying to clone the public repo (needs Internet ON)...")
    REPO_URL = "https://github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git"
    REPO_DIR = Path("/kaggle/working/NeuroMatch_2026_Behaviour")
    if not REPO_DIR.parent.exists():
        REPO_DIR = Path("./NeuroMatch_2026_Behaviour")

    if not REPO_DIR.exists():
        os.system(f"git clone {REPO_URL} {REPO_DIR}")

    matches = list(REPO_DIR.rglob("data01_direction4priors.csv"))
    if not matches:
        raise FileNotFoundError(
            "data01_direction4priors.csv not found locally and the git clone didn't work. "
            "Easiest fix: upload the CSV directly via 'Add Input > Upload' in the right panel, "
            "then re-run this cell -- no internet required that way."
        )
    CSV_PATH = matches[0]

raw = pd.read_csv(CSV_PATH)
print(f"Loaded from: {CSV_PATH}")
print(f"raw shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")


In [ ]:
data = raw.copy()

data['estimate_deg'] = np.degrees(np.arctan2(data.estimate_y, data.estimate_x)) % 360
data.loc[data.estimate_deg == 0, 'estimate_deg'] = 360

n_before = len(data)
data = data.dropna(subset=['estimate_deg', 'motion_direction']).copy()
print(f"dropped {n_before - len(data)} rows with missing estimate")

data['error_deg'] = ((data.estimate_deg - data.motion_direction + 180) % 360) - 180
data['block_id'] = data['subject_id'].astype(str) + '_' + data['run_id'].astype(str)

trial_data = data.rename(columns={
    'subject_id': 'subject',
    'trial_index': 'trial',
    'prior_std': 'prior_width',
    'motion_coherence': 'coherence',
    'motion_direction': 'true_direction',
})[['subject', 'block_id', 'trial', 'prior_width', 'coherence',
    'true_direction', 'estimate_deg', 'error_deg',
    'session_id', 'experiment_id']]

# session-boundary flag (window resets only here, not every block -- see handoff)
block_context = (
    data.groupby(['subject_id', 'run_id'], as_index=False)
    .agg(session_id=('session_id', 'first'))
    .sort_values(['subject_id', 'run_id'])
    .reset_index(drop=True)
)
block_context['prev_session_id'] = block_context.groupby('subject_id')['session_id'].shift(1)
block_context['same_session_prev'] = (
    block_context['prev_session_id'].notna()
    & (block_context['session_id'] == block_context['prev_session_id'])
)
block_context['block_id'] = block_context['subject_id'].astype(str) + '_' + block_context['run_id'].astype(str)

trial_data = trial_data.merge(
    block_context[['block_id', 'run_id', 'same_session_prev']], on='block_id', how='left'
)

SUBJECTS = sorted(trial_data['subject'].unique().tolist())
print(f"subjects found: {SUBJECTS}")
print(f"trial_data shape: {trial_data.shape}")


## 2. The reliability-mixture model and its NLL

Unchanged from Section 4 / Section 7 of the handoff notebook: a discrete mixture
(added, never multiplied) between a prior component and a likelihood component,
with `prior_reliance` updated trial-by-trial via a delta rule against a 5-trial
smoothed window of the true feedback direction (not the subject's own estimate,
to avoid circularity).


In [ ]:
DEG = np.arange(1, 361)

def vm_pdf(support_deg, mu_deg, k, norm=True):
    mu = np.atleast_1d(np.asarray(mu_deg, float))
    x = np.deg2rad(np.asarray(support_deg, float))[:, None]
    u = np.deg2rad(mu)[None, :]
    k = float(k)
    if np.isinf(k) or k > 1e300:
        out = np.zeros((len(support_deg), len(mu)))
        for j, mm in enumerate(mu):
            out[np.argmin(np.abs(np.asarray(support_deg) - mm)), j] = 1.0
    else:
        out = np.exp(k * np.cos(x - u) - k) / (2 * np.pi * i0e(k))
    if norm:
        out = out / out.sum(0, keepdims=True)
    return out


def wrap_signed_deg(diff_deg):
    return ((diff_deg + 180) % 360) - 180


def circular_mean_deg(angles_deg):
    angles_rad = np.deg2rad(np.asarray(angles_deg))
    mean_x = np.mean(np.cos(angles_rad))
    mean_y = np.mean(np.sin(angles_rad))
    return float(np.degrees(np.arctan2(mean_y, mean_x)) % 360)


def prior_agreement(measurement_deg, prior_mean_deg, k_prior):
    delta_rad = np.deg2rad(wrap_signed_deg(measurement_deg - prior_mean_deg))
    return float(np.exp(k_prior * (np.cos(delta_rad) - 1.0)))


def trial_percept_distribution(prior_mean_deg, measurement_deg, k_prior, k_llh, prior_reliance):
    prior_component = vm_pdf(DEG, prior_mean_deg, k_prior)[:, 0]
    llh_component = vm_pdf(DEG, measurement_deg, k_llh)[:, 0]
    mixture = prior_reliance * prior_component + (1 - prior_reliance) * llh_component
    return mixture / mixture.sum()


def circ_convolve_vec(p, kernel):
    return np.real(np.fft.ifft(np.fft.fft(p) * np.fft.fft(kernel)))


def hb_mixture_trial_loop(d, params, prior_mean_deg=225.0, window_size=5):
    k_llh, k_prior = params['k_llh'], params['k_prior']
    alpha, k_motor, lapse_rate = params['alpha'], params['k_motor'], params['lapse_rate']

    reliance = 0.5
    window_buf = deque(maxlen=window_size)
    prev_block_id = None
    motor_kernel = vm_pdf(DEG, 360.0, k_motor)[:, 0]

    n = len(d)
    p_obs = np.empty(n)
    reliance_trace = np.empty(n)

    block_ids = d['block_id'].to_numpy()
    same_sess = d['same_session_prev'].to_numpy()
    true_dirs = d['true_direction'].to_numpy(dtype=float)
    coherences = d['coherence'].to_numpy()
    prior_widths = d['prior_width'].to_numpy()
    est_degs = d['estimate_deg'].to_numpy(dtype=float)

    for i in range(n):
        is_new_block = block_ids[i] != prev_block_id
        if is_new_block and not bool(same_sess[i]):
            window_buf.clear()

        reliance_trace[i] = reliance
        kp = k_prior[prior_widths[i]]
        kl = k_llh[coherences[i]]

        mixture = trial_percept_distribution(prior_mean_deg, true_dirs[i], kp, kl, reliance)
        with_motor = circ_convolve_vec(mixture, motor_kernel)
        with_motor = np.clip(with_motor, 0, None)
        with_motor = with_motor / with_motor.sum()
        final = (1 - lapse_rate) * with_motor + lapse_rate * (1.0 / 360)

        e_idx = int(round(est_degs[i])) % 360
        if e_idx == 0:
            e_idx = 360
        p_obs[i] = final[e_idx - 1]

        window_buf.append(true_dirs[i])
        smoothed = circular_mean_deg(list(window_buf))
        agreement = prior_agreement(smoothed, prior_mean_deg, kp)
        reliance = float(np.clip(reliance + alpha * (agreement - reliance), 1e-4, 1 - 1e-4))

        prev_block_id = block_ids[i]

    return p_obs, reliance_trace


def hb_mixture_nll(theta, d, cohs_u, ps_u, window_size=5):
    i = 0
    k_llh = {c: float(np.exp(theta[i + j])) for j, c in enumerate(cohs_u)}; i += len(cohs_u)
    k_prior = {p: float(np.exp(theta[i + j])) for j, p in enumerate(ps_u)}; i += len(ps_u)
    alpha = 1.0 / (1.0 + np.exp(-theta[i])); i += 1
    k_motor = float(np.exp(theta[i])); i += 1
    lapse_rate = 0.3 / (1.0 + np.exp(-theta[i]))
    params = dict(k_llh=k_llh, k_prior=k_prior, alpha=alpha, k_motor=k_motor, lapse_rate=lapse_rate)
    p_obs, _ = hb_mixture_trial_loop(d, params, window_size=window_size)
    p_obs = np.clip(p_obs, 1e-320, None)
    return -np.log(p_obs).sum()


cohs_u, ps_u = [0.06, 0.12, 0.24], [80, 40, 20, 10]


## 3. The bounded 8-start fit, as a reusable function

8 starting points (one corrected baseline + seven systematically varied) and the
same optimizer settings as Section 9 of the handoff notebook, wrapped so it can
run per subject.

`K_PRIOR_UPPER` is now 1000, raised from an earlier attempt at 500: on the first
full run across subjects, more than half (5 of the first 9 completed) hit the 500
ceiling, so it wasn't just an edge case for one subject, it was systematically too
tight. 1000 is the same value validated earlier on subject 4 alone, where the true
optimum settled around ~190-198 (fit was essentially flat above ~150) and even the
degenerate-collapse runs only drifted out to 400-780 when given room to 1000. Still
worth checking the automatic `near_ceiling` flag below per subject, since we now know
the right ceiling can vary by subject.


In [ ]:
K_PRIOR_UPPER = 1000.0
N_STARTS = 8
SEED = 42

def make_starts(seed=SEED, n=N_STARTS):
    rng = np.random.default_rng(seed)
    base_k_llh = np.array([1.0, 3.0, 8.0])
    base_k_prior = np.array([0.7, 2.8, 8.7, 33.3])  # narrower prior -> higher kappa guess
    base_alpha, base_k_motor, base_lapse = 0.1, 30.0, 0.05

    starts = [(base_k_llh.copy(), base_k_prior.copy(), base_alpha, base_k_motor, base_lapse)]
    for _ in range(n - 1):
        factor_llh = rng.choice([0.2, 0.5, 2.0, 5.0], size=3)
        factor_prior = rng.choice([0.2, 0.5, 2.0, 5.0], size=4)
        starts.append((
            base_k_llh * factor_llh, base_k_prior * factor_prior,
            float(rng.uniform(0.02, 0.4)), float(rng.uniform(5, 80)), float(rng.uniform(0.01, 0.15))
        ))
    return starts


def to_theta(k_llh, k_prior, alpha, k_motor, lapse):
    return np.concatenate([np.log(k_llh), np.log(k_prior),
                            [np.log(alpha / (1 - alpha))], [np.log(k_motor)],
                            [np.log(lapse / (0.3 - lapse))]])


def decode_theta(x):
    k_llh = np.exp(x[0:3])
    k_prior = np.exp(x[3:7])
    alpha = 1 / (1 + np.exp(-x[7]))
    k_motor = np.exp(x[8])
    lapse = 0.3 / (1 + np.exp(-x[9]))
    return k_llh, k_prior, alpha, k_motor, lapse


def bounds_with_ceiling(k_prior_upper):
    return (
        [(np.log(0.05), np.log(150))] * 3                 # k_llh x3
        + [(np.log(0.05), np.log(k_prior_upper))] * 4       # k_prior x4
        + [(np.log(0.01 / 0.99), np.log(0.6 / 0.4))]        # alpha
        + [(np.log(1), np.log(400))]                        # k_motor
        + [(np.log(0.001 / 0.299), np.log(0.25 / 0.05))]    # lapse
    )


def fit_subject_bounded(subject_id, d_subject, k_prior_upper=K_PRIOR_UPPER, n_starts=N_STARTS, seed=SEED):
    """Runs the bounded 10-start Powell fit for one subject. Returns a list of
    (start_id, success, nll, theta) tuples, sorted by NLL ascending."""
    def obj(theta):
        return hb_mixture_nll(theta, d_subject, cohs_u, ps_u)

    starts = make_starts(seed=seed, n=n_starts)
    bounds = bounds_with_ceiling(k_prior_upper)
    results = []
    for start_id, (k_llh0, k_prior0, alpha0, k_motor0, lapse0) in enumerate(starts, 1):
        k_llh0 = np.clip(k_llh0, 0.05, 150)
        k_prior0 = np.clip(k_prior0, 0.05, k_prior_upper)
        alpha0 = float(np.clip(alpha0, 0.01, 0.6))
        k_motor0 = float(np.clip(k_motor0, 1, 400))
        lapse0 = float(np.clip(lapse0, 0.001, 0.25))
        theta0 = to_theta(k_llh0, k_prior0, alpha0, k_motor0, lapse0)
        theta0 = np.clip(theta0, [b[0] for b in bounds], [b[1] for b in bounds])
        t0 = time.time()
        res = minimize(obj, theta0, method="Powell", bounds=bounds,
                       options=dict(maxiter=60, xtol=5e-3, ftol=5e-3))
        dt = time.time() - t0
        results.append((start_id, bool(res.success), float(res.fun), res.x.copy()))
        print(f"  subject {subject_id} | start {start_id:2d}: success={res.success} "
              f"NLL={res.fun:.1f}  ({dt:.0f}s)")
    return sorted(results, key=lambda r: r[2])


## 4a. (Chunked commits only) Recover checkpoints from a previous commit

Skip this if you're doing Option A (one big commit) -- there won't be
anything to recover on a first run anyway, and this cell is harmless to run
either way.

If this is a follow-up chunk: after your previous commit finished, go to that
version's Output tab and click **"New Dataset"** to publish its output files
(the `hb_fits/` folder) as a Kaggle dataset. Then, on this notebook, use
**Add Input** to attach that dataset before running this cell. It copies any
`subject_*.pkl` checkpoints it finds among your attached inputs into this
session's `OUT_DIR`, so subjects already fit in earlier chunks are recognized
and skipped below, and included in the final summary.


In [ ]:
import shutil

prior_checkpoints = glob.glob("/kaggle/input/**/subject_*.pkl", recursive=True)
recovered = 0
for p in prior_checkpoints:
    dest = OUT_DIR / Path(p).name
    if not dest.exists():
        shutil.copy(p, dest)
        recovered += 1

print(f"Recovered {recovered} checkpoint(s) from previously attached input dataset(s).")
already_done = sorted(int(p.stem.split('_')[1]) for p in OUT_DIR.glob("subject_*.pkl"))
print(f"Subjects with an existing checkpoint right now: {already_done}")


## 4b. Choose which subjects this commit will fit

Edit `SUBJECTS_TO_RUN` below before each commit.

**Important:** don't chunk sequentially by subject ID. Trial counts per subject
range from 4,799 to 9,412 (almost 2x), and fit time scales with trial count, so a
naive `[1,2,3,4]` / `[5,6,7,8]` / `[9,10,11,12]` split can accidentally front-load
the heaviest subjects into one chunk. The pairing below balances each chunk's
total trial count instead (largest subject paired with smallest, etc.):

| Chunk | Subjects | Total trials |
|---|---|---|
| 1 | `[3, 4]`   | 14,211 |
| 2 | `[9, 5]`   | 14,421 |
| 3 | `[1, 7]`   | 14,359 |
| 4 | `[2, 8]`   | 13,674 |
| 5 | `[6, 10]`  | 13,597 |
| 6 | `[11, 12]` | 12,948 |

**Time estimate, recalibrated from the actual first attempt's log** (much more
reliable than an estimate from subject 4 alone): with `N_STARTS=8`, each 2-subject
chunk above takes roughly **~2 hours**, not the ~1-1.5 hours originally guessed. All
12 subjects done as one continuous run would be closer to 11-12 hours, which is
almost certainly why the original single-commit attempt hit the wall without
finishing. The chunked route stays the safer bet.

For Option A (one big commit), just leave this as `SUBJECTS` (all 12).


In [ ]:
# --- EDIT THIS before each commit if you're chunking ---
SUBJECTS_TO_RUN = SUBJECTS  # <- change to e.g. [1, 2, 3, 4] for a chunk

print(f"This commit will attempt to fit: {SUBJECTS_TO_RUN}")


## 4c. Main loop: fit `SUBJECTS_TO_RUN`, with checkpointing

Only fits subjects listed in `SUBJECTS_TO_RUN` (Section 4b). Skips any subject whose checkpoint file already exists in `OUT_DIR`, so re-running
this cell after an interruption picks up where it left off. Each checkpoint stores:
the full multi-start table (for the same transparency check as Section 8),
the best-fit parameters, NLL, convergence flag, and (computed at the best-fit params)
the per-trial predicted probability of the observed response and the reliance
trajectory -- the "predicted distributions" the original task asked for, kept at
per-trial resolution rather than the full 360-bin distribution per trial (which
would be tens of GB across all subjects and trials).


In [ ]:
def checkpoint_path(subject_id):
    return OUT_DIR / f"subject_{subject_id}.pkl"


for subject_id in SUBJECTS_TO_RUN:
    ckpt = checkpoint_path(subject_id)
    if ckpt.exists():
        print(f"subject {subject_id}: checkpoint already exists, skipping")
        continue

    print(f"\n=== fitting subject {subject_id} ===")
    d_subject = trial_data[trial_data.subject == subject_id].sort_values(['run_id', 'trial']).reset_index(drop=True)
    print(f"  {len(d_subject)} trials")

    t_start = time.time()
    results = fit_subject_bounded(subject_id, d_subject)
    elapsed = time.time() - t_start

    best_start_id, best_success, best_nll, best_theta = results[0]
    k_llh, k_prior, alpha, k_motor, lapse = decode_theta(best_theta)

    # flag if the best fit is still hugging the k_prior ceiling, or looks like
    # the degenerate alpha-collapse mode described in the handoff
    near_ceiling = bool(np.any(k_prior > 0.9 * K_PRIOR_UPPER))
    likely_degenerate = bool(alpha < 0.02)

    params_best = dict(
        k_llh=k_llh, k_prior=k_prior, alpha=alpha, k_motor=k_motor, lapse_rate=lapse
    )
    params_best_named = dict(
        k_llh={c: float(v) for c, v in zip(cohs_u, k_llh)},
        k_prior={p: float(v) for p, v in zip(ps_u, k_prior)},
        alpha=alpha, k_motor=k_motor, lapse_rate=lapse,
    )
    p_obs, reliance_trace = hb_mixture_trial_loop(
        d_subject,
        dict(k_llh=params_best_named['k_llh'], k_prior=params_best_named['k_prior'],
             alpha=alpha, k_motor=k_motor, lapse_rate=lapse),
    )

    checkpoint = dict(
        subject_id=subject_id,
        n_trials=len(d_subject),
        all_starts=results,               # full 10-start table, sorted by NLL
        best_start_id=best_start_id,
        best_success=best_success,
        best_nll=best_nll,
        best_params=params_best_named,
        near_ceiling=near_ceiling,
        likely_degenerate=likely_degenerate,
        k_prior_upper_used=K_PRIOR_UPPER,
        p_obs=p_obs,
        reliance_trace=reliance_trace,
        fit_time_seconds=elapsed,
    )
    with open(ckpt, "wb") as f:
        pickle.dump(checkpoint, f)

    flag = ""
    if near_ceiling:
        flag += "  [WARNING: near k_prior ceiling, consider raising K_PRIOR_UPPER]"
    if likely_degenerate:
        flag += "  [WARNING: looks like the degenerate alpha-collapse mode]"
    print(f"  done in {elapsed:.0f}s | best NLL={best_nll:.1f} | alpha={alpha:.3f}{flag}")

print("\nAll subjects processed (or already checkpointed).")


## 5. Summary across subjects

Loads every checkpoint and builds one table: best NLL, `alpha`, `k_prior` per
condition, convergence, and the two automatic warning flags. This is the input
to the next open items: correlating `alpha` against `pull_increase` (item 3),
and investigating the non-monotonic `k_prior` pattern (item 4/7).


In [ ]:
rows = []
for ckpt_file in sorted(OUT_DIR.glob("subject_*.pkl")):
    with open(ckpt_file, "rb") as f:
        c = pickle.load(f)
    kp = c['best_params']['k_prior']
    kl = c['best_params']['k_llh']
    rows.append(dict(
        subject=c['subject_id'],
        n_trials=c['n_trials'],
        best_nll=c['best_nll'],
        nll_per_trial=c['best_nll'] / c['n_trials'],
        converged=c['best_success'],
        alpha=c['best_params']['alpha'],
        k_motor=c['best_params']['k_motor'],
        lapse_rate=c['best_params']['lapse_rate'],
        k_prior_80=kp[80], k_prior_40=kp[40], k_prior_20=kp[20], k_prior_10=kp[10],
        k_llh_06=kl[0.06], k_llh_12=kl[0.12], k_llh_24=kl[0.24],
        near_ceiling=c['near_ceiling'],
        likely_degenerate=c['likely_degenerate'],
        fit_time_seconds=c['fit_time_seconds'],
    ))

summary = pd.DataFrame(rows).sort_values('subject').reset_index(drop=True)
summary_path = OUT_DIR / "summary_all_subjects.csv"
summary.to_csv(summary_path, index=False)

print(f"saved summary to {summary_path}")
print(f"\n{len(summary)}/12 subjects fitted so far")
n_flagged = summary['near_ceiling'].sum() + summary['likely_degenerate'].sum()
if n_flagged > 0:
    print(f"WARNING: {int(n_flagged)} subject(s) flagged, check 'near_ceiling' / 'likely_degenerate' columns")
summary
